In [ ]:
# =====================================================================
#
#        GENERIC DATA CLEANING PIPELINE  —  V2
#        Pipeline Générique de Nettoyage de Données
#
# =====================================================================
#
# AUTEUR     : Fahim Coulibaly AI / ML Engineer
# VERSION    : 2.0
# LANGAGE    : Python 3.9+
#
# DÉPENDANCES
# ---------------------------------------------------------------------
#
#   pip install pandas scikit-learn numpy
#
# =====================================================================


# =====================================================================
# PHILOSOPHIE DU PIPELINE
# =====================================================================
#
#   Comprendre
#        ↓
#   Nettoyer
#        ↓
#   Vérifier
#        ↓
#   Analyser
#        ↓
#   Interpréter
#
# ---------------------------------------------------------------------
#
# COMPRENDRE
#
#   Avant de toucher aux données, il faut les observer.
#   Combien de lignes ? De colonnes ? Quels types ?
#   Quelles valeurs manquent ?
#
# NETTOYER
#
#   Supprimer ce qui n'apporte rien (doublons, colonnes constantes).
#   Corriger ce qui est faux (types, valeurs aberrantes).
#   Combler intelligemment les trous (imputation).
#
# VÉRIFIER
#
#   Après chaque transformation, on vérifie.
#   Rien n'est assumé. Tout est confirmé.
#
# ANALYSER
#
#   Le dataset est maintenant prêt.
#   On peut explorer, visualiser, modéliser.
#
# INTERPRÉTER
#
#   Les chiffres seuls ne suffisent pas.
#   Un analyste comprend le contexte métier
#   et donne du sens aux résultats.
#
# =====================================================================


# =====================================================================
# COMMENT PENSER ? — GUIDE DE DÉCISION
# =====================================================================
#
# Quand tu inspectes une colonne, pose-toi ces questions :
#
# ┌──────────────────────┬──────────────────────────────────────────┐
# │ TYPE CIBLE           │ QUESTIONS À SE POSER                     │
# ├──────────────────────┼──────────────────────────────────────────┤
# │ Numérique            │ Est-ce une mesure ? Un montant ?         │
# │ (int, float)         │ Peut-on calculer une moyenne dessus ?    │
# │                      │ Ex : age, salary, temperature            │
# ├──────────────────────┼──────────────────────────────────────────┤
# │ Date                 │ Est-ce un moment dans le temps ?         │
# │ (datetime)           │ Ex : order_date, created_at              │
# ├──────────────────────┼──────────────────────────────────────────┤
# │ Catégorie            │ Est-ce un groupe limité de valeurs ?     │
# │ (category)           │ Ex : gender, country, product_type       │
# ├──────────────────────┼──────────────────────────────────────────┤
# │ Booléen              │ Est-ce une valeur vrai/faux ?            │
# │ (bool)               │ Ex : is_active, has_paid, churn          │
# ├──────────────────────┼──────────────────────────────────────────┤
# │ Identifiant          │ Identifie-t-on une entité unique ?       │
# │ (string, conserver)  │ Ex : customer_id, zipcode, product_code  │
# └──────────────────────┴──────────────────────────────────────────┘
#
# CAS PIÈGES — IDENTIFIANTS NUMÉRIQUES
# ---------------------------------------------------------------------
#
# Ces colonnes SEMBLENT numériques mais NE LE SONT PAS :
#
#   customer_id   → 10042, 10043...
#   employee_id   → 98001, 98002...
#   zipcode       → 75001, 75002...
#   product_code  → 4001, 4002...
#   phone_number  → 0612345678
#
# Pourquoi ne pas les convertir en numérique ?
#
#   1. On n'additionne jamais deux customer_id.
#   2. On ne calcule pas la moyenne des zipcodes.
#   3. Ce sont des LABELS, pas des MESURES.
#   4. Convertir en float introduit des .0 parasites.
#   5. On risque des tronquages (int64 overflow).
#
# → Règle : si tu ne feras jamais d'arithmétique dessus, garde-le en string.
#
# =====================================================================


# =====================================================================
# WORKFLOW GLOBAL
# =====================================================================
#
#   RAW DATASET
#        │
#        ▼
#   ① inspect_dataframe()
#        │
#        ▼
#   ② handle_duplicates()
#        │
#        ▼
#   ③ handle_missing_values()
#        │
#        ▼
#   ④ drop_constant_columns()
#        │
#        ▼
#   ⑤ detect_outliers_iqr()       ← rapport uniquement, pas de suppression
#        │
#        ▼
#   ⑥ clean_numeric_strings()
#        │
#        ▼
#   ⑦ detect_column_types()
#   ⑦ detect_boolean_columns()
#        │
#        ▼
#   ⑧ validate_detected_types()   ← validation humaine obligatoire
#        │
#        ▼
#   ⑨ convert_column_types()
#   ⑨ convert_boolean_columns()
#        │
#        ▼
#   ⑩ optimize_numeric_types()
#        │
#        ▼
#   ⑪ final_validation()
#        │
#        ▼
#   CLEAN DATASET
#
# =====================================================================


# =====================================================================
# IMPORTS
# =====================================================================

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer


# =====================================================================
# 1. INSPECTION
# =====================================================================
#
# POURQUOI CETTE ÉTAPE ?
#
#   Avant de modifier quoi que ce soit,
#   il faut comprendre ce qu'on a entre les mains.
#
# CE QU'ELLE RÉSOUT :
#
#   - Révèle la structure réelle du dataset.
#   - Identifie les colonnes problématiques dès le départ.
#   - Oriente les décisions de nettoyage qui suivent.
#
# SI ON LA SAUTE :
#
#   On travaille à l'aveugle. On risque de transformer
#   des données sans comprendre leur nature réelle.
#   Les erreurs sont découvertes trop tard.
#
# =====================================================================

def inspect_dataframe(df):
    """
    Affiche un rapport complet du dataset brut.

    Informations présentées :
        - Dimensions (lignes × colonnes)
        - Types de chaque colonne
        - Nombre et pourcentage de valeurs manquantes
        - Utilisation mémoire
        - Aperçu des premières lignes

    Parameters
    ----------
    df : pd.DataFrame
        Le dataset brut à inspecter.

    Returns
    -------
    None
        Affiche uniquement, ne modifie pas le dataset.

    Example
    -------
    >>> df = pd.read_csv("data.csv")
    >>> inspect_dataframe(df)
    """

    print("\n" + "=" * 60)
    print("DATASET OVERVIEW")
    print("=" * 60)

    # Dimensions
    print(f"\nRows    : {df.shape[0]:,}")
    print(f"Columns : {df.shape[1]}")

    # Types des colonnes
    print("\n--- Data Types ---")
    print(df.dtypes)

    # Valeurs manquantes : count ET pourcentage
    print("\n--- Missing Values ---")
    missing = pd.DataFrame({
        "count"  : df.isnull().sum(),
        "percent": round(df.isnull().mean() * 100, 2)
    })
    # On affiche seulement les colonnes qui ont des manquants
    missing_only = missing[missing["count"] > 0]
    if missing_only.empty:
        print("Aucune valeur manquante détectée.")
    else:
        print(missing_only)

    # Mémoire utilisée par le dataset
    print("\n--- Memory Usage ---")
    memory_mb = (
        df.memory_usage(deep=True)   # deep=True pour les strings
        .sum()
        / 1024 ** 2                  # octets → mégaoctets
    )
    print(f"{memory_mb:.2f} MB")

    # Aperçu des premières lignes
    print("\n--- Preview (first 5 rows) ---")
    print(df.head())


# =====================================================================
# 2. DUPLICATES
# =====================================================================
#
# POURQUOI CETTE ÉTAPE ?
#
#   Les doublons sont des lignes identiques répétées.
#   Ils apparaissent souvent à cause de :
#   - jointures mal maîtrisées
#   - collectes multiples
#   - erreurs d'import
#
# CE QU'ILS CASSENT :
#
#   - Les moyennes (une valeur compte double)
#   - Les modèles ML (l'observation est surpondérée)
#   - Les graphiques (fausses distributions)
#   - Les KPIs métier (chiffres gonflés)
#
# Exemple :
#
#   client_id   amount
#   1001        500      ← ligne originale
#   1001        500      ← doublon → la moyenne est faussée
#
# SI ON LA SAUTE :
#
#   Des observations sont comptées plusieurs fois.
#   Les statistiques et modèles sont biaisés.
#
# =====================================================================

def inspect_duplicates(df):
    """
    Compte et affiche le nombre de lignes dupliquées.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    int
        Nombre de lignes dupliquées.
    """

    # duplicated() retourne True pour chaque ligne
    # qui apparaît déjà au-dessus dans le DataFrame
    duplicates = df.duplicated().sum()

    print("\n" + "=" * 60)
    print("DUPLICATES REPORT")
    print("=" * 60)
    print(f"\nDuplicates found : {duplicates:,}")

    return duplicates


def show_duplicates(df):
    """
    Retourne uniquement les lignes dupliquées pour inspection.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
        Sous-ensemble contenant les doublons.
    """
    return df[df.duplicated()]


def remove_duplicates(df):
    """
    Supprime les lignes dupliquées et affiche le bilan.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
        Dataset sans doublons.
    """

    before = len(df)

    # drop_duplicates() conserve la première occurrence
    # et supprime toutes les suivantes identiques
    df = df.drop_duplicates()

    after = len(df)

    print(f"\n{before - after:,} duplicate(s) removed.")
    print(f"Rows remaining : {after:,}")

    return df


def handle_duplicates(df):
    """
    Orchestre la détection et la suppression des doublons.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
        Dataset sans doublons.
    """

    inspect_duplicates(df)
    df = remove_duplicates(df)

    return df


# =====================================================================
# 3. MISSING VALUES
# =====================================================================
#
# POURQUOI CETTE ÉTAPE ?
#
#   Les données réelles sont rarement complètes.
#   Des valeurs manquantes (NaN) viennent de :
#   - champs non remplis dans un formulaire
#   - erreurs de collecte
#   - colonnes inapplicables pour certains individus
#
# STRATÉGIE ADOPTÉE
# ---------------------------------------------------------------------
#
#   Missing > 50%
#       → Supprimer la colonne
#         (trop peu d'information pour être utile)
#
#   5% < Missing ≤ 50%
#       → Imputation
#         Numérique   → Médiane (robuste aux outliers)
#         Catégoriel  → Mode (valeur la plus fréquente)
#
#   Missing ≤ 5%
#       → Supprimer les lignes
#         (faible impact, évite toute hypothèse)
#
# POURQUOI LA MÉDIANE ET PAS LA MOYENNE ?
#
#   La moyenne est sensible aux valeurs aberrantes.
#   Ex : salaires [2000, 2200, 2100, 150000]
#   Moyenne ≈ 39075 (biaisée)
#   Médiane = 2150  (représentative)
#
# SI ON LA SAUTE :
#
#   La plupart des algorithmes ML rejettent les NaN.
#   Les statistiques sont faussées ou impossibles à calculer.
#
# =====================================================================

def inspect_missing_values(df):
    """
    Génère un rapport des valeurs manquantes par colonne.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
        Rapport trié par taux de manquants décroissant.
        Colonnes : missing_count, missing_percent
    """

    report = pd.DataFrame({
        "missing_count"  : df.isnull().sum(),
        "missing_percent": round(df.isnull().mean() * 100, 2)
    })

    return report.sort_values(
        by="missing_percent",
        ascending=False
    )


def drop_high_missing_columns(df, threshold=50):
    """
    Supprime les colonnes avec plus de X% de valeurs manquantes.

    Parameters
    ----------
    df : pd.DataFrame
    threshold : float, default=50
        Seuil en pourcentage au-delà duquel la colonne est supprimée.

    Returns
    -------
    pd.DataFrame
    """

    # Calcul du taux de manquants pour chaque colonne
    missing_percent = df.isnull().mean() * 100

    # Colonnes dépassant le seuil
    cols_to_drop = missing_percent[
        missing_percent > threshold
    ].index.tolist()

    if cols_to_drop:
        print(f"\nColumns dropped (>{threshold}% NaN) :")
        for col in cols_to_drop:
            print(f"  - {col}")
        df = df.drop(columns=cols_to_drop)
    else:
        print(f"\nNo column exceeded {threshold}% missing threshold.")

    return df


def drop_low_missing_rows(df, threshold=5):
    """
    Supprime les lignes dont les colonnes ont ≤ X% de manquants.

    Ces colonnes sont "presque complètes" : supprimer les quelques
    lignes concernées a peu d'impact sur le dataset global.

    Parameters
    ----------
    df : pd.DataFrame
    threshold : float, default=5

    Returns
    -------
    pd.DataFrame
    """

    missing_percent = df.isnull().mean() * 100

    # Colonnes concernées par ce seuil bas
    cols = missing_percent[
        missing_percent <= threshold
    ].index.tolist()

    before = len(df)

    # Suppression des lignes qui ont un NaN dans ces colonnes
    df = df.dropna(subset=cols)

    after = len(df)

    print(f"\n{before - after:,} row(s) dropped (columns with ≤{threshold}% NaN).")

    return df


def impute_medium_missing_columns(df, lower=5, upper=50):
    """
    Impute les colonnes dont le taux de manquants est entre lower% et upper%.

    Stratégie :
        Numérique   → Médiane
        Catégoriel  → Mode (most_frequent)

    Parameters
    ----------
    df : pd.DataFrame
    lower : float, default=5
    upper : float, default=50

    Returns
    -------
    pd.DataFrame
    """

    missing_percent = df.isnull().mean() * 100

    # Colonnes dans la fourchette intermédiaire
    cols = missing_percent[
        (missing_percent > lower) &
        (missing_percent <= upper)
    ].index.tolist()

    if not cols:
        print("\nNo column required imputation.")
        return df

    print("\nColumns imputed :")

    for col in cols:

        if pd.api.types.is_numeric_dtype(df[col]):
            # Médiane : robuste aux valeurs extrêmes
            strategy = "median"
        else:
            # Mode : valeur la plus fréquente
            strategy = "most_frequent"

        imputer = SimpleImputer(strategy=strategy)

        # fit_transform attend un tableau 2D → double crochet [[col]]
        df[[col]] = imputer.fit_transform(df[[col]])

        print(f"  - {col} ({strategy})")

    return df


def handle_missing_values(df):
    """
    Orchestre le traitement complet des valeurs manquantes.

    Étapes :
        1. Rapport des manquants
        2. Suppression colonnes > 50%
        3. Suppression lignes pour colonnes ≤ 5%
        4. Imputation pour colonnes entre 5% et 50%

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """

    print("\n" + "=" * 60)
    print("MISSING VALUES REPORT")
    print("=" * 60)

    print(inspect_missing_values(df))

    df = drop_high_missing_columns(df)
    df = drop_low_missing_rows(df)
    df = impute_medium_missing_columns(df)

    return df


# =====================================================================
# 4. COLONNES CONSTANTES
# =====================================================================
#
# POURQUOI CETTE ÉTAPE ?
#
#   Une colonne constante a la même valeur pour toutes les lignes.
#   Elle n'apporte aucune information discriminante.
#
# Exemples :
#
#   country = "France"  pour 100% des lignes
#   version = "1.0"     pour 100% des lignes
#
# CE QU'ELLES CASSENT :
#
#   - Variance = 0 → division par zéro dans certains algo
#   - Corrélations impossibles à calculer
#   - Confusion dans les modèles ML
#   - Espace mémoire inutile
#
# SI ON LA SAUTE :
#
#   Certains algorithmes (StandardScaler, PCA, LDA)
#   génèrent des erreurs ou des résultats absurdes.
#
# =====================================================================

def drop_constant_columns(df):
    """
    Supprime les colonnes dont toutes les valeurs sont identiques.

    Une colonne est constante si nunique() == 1
    (une seule valeur unique dans toute la colonne).

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
        Dataset sans colonnes constantes.

    Example
    -------
    Avant : country = ['FR', 'FR', 'FR', 'FR']
    Après : colonne supprimée
    """

    print("\n" + "=" * 60)
    print("CONSTANT COLUMNS")
    print("=" * 60)

    # nunique() compte le nombre de valeurs distinctes par colonne
    # dropna=False : on compte aussi NaN comme une valeur distincte
    constant_cols = [
        col for col in df.columns
        if df[col].nunique(dropna=False) <= 1
    ]

    if constant_cols:
        print(f"\nConstant columns removed ({len(constant_cols)}) :")
        for col in constant_cols:
            print(f"  - {col}  →  unique value : {df[col].unique()}")
        df = df.drop(columns=constant_cols)
    else:
        print("\nNo constant column detected.")

    return df


# =====================================================================
# 5. DÉTECTION DES OUTLIERS (IQR)
# =====================================================================
#
# POURQUOI CETTE ÉTAPE ?
#
#   Les outliers (valeurs aberrantes) sont des observations
#   très éloignées du reste des données.
#
# Exemples :
#
#   salaire = [2000, 2100, 2200, 2050, 500000]  ← 500000 est suspect
#   age     = [25, 30, 28, 32, -5]              ← -5 impossible
#
# MÉTHODE IQR (Interquartile Range)
#
#   Q1 = 25ème percentile
#   Q3 = 75ème percentile
#   IQR = Q3 - Q1
#
#   Borne inférieure = Q1 - 1.5 × IQR
#   Borne supérieure = Q3 + 1.5 × IQR
#
#   Toute valeur hors de ces bornes est un outlier potentiel.
#
# POURQUOI ON NE SUPPRIME PAS AUTOMATIQUEMENT ?
#
#   L'algorithme détecte des anomalies STATISTIQUES.
#   Mais l'analyste doit décider du sens MÉTIER :
#
#   → Un salaire de 500 000 € peut être le PDG de l'entreprise.
#     C'est statistiquement aberrant, mais RÉEL et VALIDE.
#
#   → Un âge de -5 ans est statistiquement ET logiquement faux.
#     On peut le corriger ou supprimer.
#
#   La machine ne peut pas faire cette distinction.
#   L'humain le doit.
#
# =====================================================================

def detect_outliers_iqr(df, factor=1.5):
    """
    Détecte les outliers dans les colonnes numériques via la méthode IQR.

    Génère un rapport détaillé SANS supprimer aucune valeur.
    La décision de traitement appartient à l'analyste.

    Méthode IQR :
        Q1      = 25ème percentile
        Q3      = 75ème percentile
        IQR     = Q3 - Q1
        Borne basse = Q1 - factor * IQR
        Borne haute = Q3 + factor * IQR

    Parameters
    ----------
    df : pd.DataFrame
    factor : float, default=1.5
        Multiplicateur IQR. 1.5 = standard, 3.0 = outliers extrêmes.

    Returns
    -------
    pd.DataFrame
        Rapport avec colonnes :
        col, q1, q3, iqr, lower_bound, upper_bound,
        outlier_count, outlier_percent
    """

    print("\n" + "=" * 60)
    print("OUTLIERS REPORT (IQR Method)")
    print("=" * 60)
    print(
        "\n⚠ Ce rapport est informatif uniquement.\n"
        "  Aucune valeur n'est supprimée automatiquement.\n"
        "  L'analyste doit valider chaque cas."
    )

    # On ne traite que les colonnes numériques
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    report_rows = []

    for col in numeric_cols:

        # Calcul des quartiles
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        # Bornes de détection
        lower = q1 - factor * iqr
        upper = q3 + factor * iqr

        # Masque booléen : True si la valeur est hors bornes
        outlier_mask = (df[col] < lower) | (df[col] > upper)
        outlier_count = outlier_mask.sum()
        outlier_pct = round(outlier_mask.mean() * 100, 2)

        report_rows.append({
            "column"         : col,
            "q1"             : round(q1, 4),
            "q3"             : round(q3, 4),
            "iqr"            : round(iqr, 4),
            "lower_bound"    : round(lower, 4),
            "upper_bound"    : round(upper, 4),
            "outlier_count"  : outlier_count,
            "outlier_percent": outlier_pct,
        })

    report = pd.DataFrame(report_rows)
    report = report.sort_values("outlier_count", ascending=False)

    print("\n")
    print(report.to_string(index=False))

    print(
        "\n→ Actions possibles par l'analyste :\n"
        "  - Supprimer les lignes outliers\n"
        "  - Remplacer par NaN puis imputer\n"
        "  - Conserver (outlier réel et valide)\n"
        "  - Créer une colonne flag  is_outlier_{col}"
    )

    return report


# =====================================================================
# 6. NETTOYAGE DES NOMBRES TEXTE
# =====================================================================
#
# POURQUOI CETTE ÉTAPE ?
#
#   Dans les datasets réels, les montants sont souvent
#   stockés comme du texte avec des symboles parasites.
#
# Exemples courants :
#
#   "$1,500"   → 1500.0
#   "2 500"    → 2500.0
#   "30%"      → 30.0
#   "€4,200"   → 4200.0
#   "1.500,00" → 1500.0
#
# CE QU'ILS EMPÊCHENT :
#
#   - Conversion en float impossible
#   - Calculs de moyennes / sommes impossibles
#   - Graphiques mal construits
#
# SI ON LA SAUTE :
#
#   Les colonnes restent en object.
#   Toute tentative de calcul numérique échoue.
#
# =====================================================================

def clean_numeric_strings(df, columns):
    """
    Nettoie les colonnes texte contenant des nombres formatés
    et les convertit en float.

    Symboles supprimés : $ € £ ¥ % , espaces insécables
    Gère les formats européens (1.500,00 → 1500.0)

    Parameters
    ----------
    df : pd.DataFrame
    columns : list of str
        Noms des colonnes à nettoyer.

    Returns
    -------
    pd.DataFrame

    Example
    -------
    Avant : price = ["$1,500", "€2 000", "30%"]
    Après : price = [1500.0, 2000.0, 30.0]
    """

    print("\n" + "=" * 60)
    print("NUMERIC STRING CLEANING")
    print("=" * 60)

    for col in columns:

        if col not in df.columns:
            print(f"  ⚠ Colonne '{col}' introuvable, ignorée.")
            continue

        original_dtype = df[col].dtype

        # Conversion en string pour manipulation textuelle
        cleaned = df[col].astype(str)

        # Suppression des symboles monétaires et %
        cleaned = cleaned.str.replace(
            r"[$€£¥%]", "", regex=True
        )

        # Suppression des espaces (séparateurs de milliers)
        cleaned = cleaned.str.replace(
            r"\s+", "", regex=True
        )

        # Gestion du format européen : 1.500,00 → 1500.00
        # Condition : présence d'un point ET d'une virgule
        has_both = cleaned.str.contains(r"\.\d{3},", regex=True)
        cleaned = cleaned.where(
            ~has_both,
            cleaned.str.replace(".", "", regex=False)
                       .str.replace(",", ".", regex=False)
        )

        # Suppression des virgules restantes (milliers anglophones)
        cleaned = cleaned.str.replace(",", "", regex=False)

        # Conversion finale en numérique
        # errors="coerce" transforme les invalides en NaN
        df[col] = pd.to_numeric(cleaned, errors="coerce")

        print(
            f"  - {col} : {original_dtype} → float64"
            f"  ({df[col].isna().sum()} NaN générés)"
        )

    return df


# =====================================================================
# 7. DÉTECTION AUTOMATIQUE DES TYPES
# =====================================================================
#
# POURQUOI CETTE ÉTAPE ?
#
#   Pandas lit tout en string par défaut si les données sont mixtes.
#   Il faut détecter le vrai type de chaque colonne.
#
# MÉTHODE
#
#   On essaie de convertir chaque colonne.
#   Si ≥ 80% des valeurs se convertissent → on valide le type.
#
# LIMITE DE L'AUTOMATISATION
#
#   L'algorithme est aveugle au contexte métier.
#   Il ne sait pas que "75001" est un code postal,
#   pas un entier qu'on additionne.
#
#   → La validation humaine (étape suivante) est indispensable.
#
# =====================================================================

def detect_numeric_columns(df, threshold=0.80):
    """
    Détecte les colonnes qui peuvent être converties en numérique.

    Un seuil de 80% signifie :
    "au moins 80% des valeurs non-nulles se convertissent en nombre".

    Parameters
    ----------
    df : pd.DataFrame
    threshold : float, default=0.80
        Taux minimal de conversions réussies.

    Returns
    -------
    list of str
    """

    numeric_cols = []

    for col in df.columns:

        # errors="coerce" : si conversion impossible → NaN
        converted = pd.to_numeric(df[col], errors="coerce")

        # Taux de succès : valeurs non-NaN après conversion
        success_rate = converted.notna().mean()

        if success_rate >= threshold:
            numeric_cols.append(col)

    return numeric_cols


def detect_date_columns(df, threshold=0.80):
    """
    Détecte les colonnes qui peuvent être converties en datetime.

    Parameters
    ----------
    df : pd.DataFrame
    threshold : float, default=0.80

    Returns
    -------
    list of str
    """

    date_cols = []

    for col in df.columns:

        # infer_datetime_format : Pandas devine le format automatiquement
        converted = pd.to_datetime(df[col], errors="coerce")

        success_rate = converted.notna().mean()

        if success_rate >= threshold:
            date_cols.append(col)

    return date_cols


def detect_category_columns(df, max_unique_ratio=0.10):
    """
    Détecte les colonnes texte avec peu de valeurs uniques.

    Une colonne catégorielle a un petit nombre de modalités
    par rapport au nombre total de lignes.

    Exemple :
        gender     → ["M", "F"]           → 2 uniques / 1000 lignes = 0.2%
        country    → 50 uniques / 1000    → 5%
        customer_id → 1000 uniques / 1000 → 100%  (identifiant, pas catégorie)

    Parameters
    ----------
    df : pd.DataFrame
    max_unique_ratio : float, default=0.10
        Seuil maximum du ratio unique/total.

    Returns
    -------
    list of str
    """

    category_cols = []

    # On ne regarde que les colonnes texte (object)
    object_cols = df.select_dtypes(include="object")

    for col in object_cols:

        # Ratio = nombre de valeurs distinctes / nombre de lignes
        ratio = df[col].nunique() / len(df)

        if ratio <= max_unique_ratio:
            category_cols.append(col)

    return category_cols


def detect_boolean_columns(df):
    """
    Détecte les colonnes qui contiennent des valeurs booléennes.

    Valeurs reconnues (insensible à la casse) :
        - True / False
        - Yes / No
        - Y / N
        - 1 / 0

    Une colonne est booléenne si elle ne contient
    QUE ces valeurs (hors NaN).

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    list of str
        Colonnes candidates pour conversion en bool.
    """

    # Ensembles de valeurs valides pour chaque encodage
    BOOL_SETS = [
        {"true", "false"},
        {"yes", "no"},
        {"y", "n"},
        {"1", "0"},
        {1, 0},
        {True, False},
    ]

    bool_cols = []

    for col in df.columns:

        # Valeurs uniques non-nulles, en minuscules si string
        unique_vals = df[col].dropna().unique()

        try:
            # Normalisation en minuscule pour la comparaison
            normalized = set(
                str(v).strip().lower() for v in unique_vals
            )
        except Exception:
            continue

        # On vérifie si l'ensemble correspond à un encodage booléen
        for bool_set in BOOL_SETS:
            normalized_bool_set = set(
                str(v).strip().lower() for v in bool_set
            )
            if normalized == normalized_bool_set:
                bool_cols.append(col)
                break

    return bool_cols


# =====================================================================
# 8. VALIDATION HUMAINE
# =====================================================================
#
# POURQUOI L'AUTOMATISATION NE SUFFIT PAS ?
#
#   L'algorithme de détection est purement statistique.
#   Il ne connaît pas votre domaine métier.
#
# EXEMPLES DE PIÈGES :
#
#   zipcode    → détecté comme numérique
#               MAIS ce n'est pas un nombre, c'est un code.
#               75001 + 75002 n'a aucun sens.
#
#   customer_id → détecté comme numérique
#               MAIS c'est un identifiant unique, pas une mesure.
#
#   phone_number → détecté comme numérique
#               MAIS doit rester en string (avec le 0 initial).
#
#   is_active → détecté comme catégoriel (si text)
#               MAIS devrait être un booléen.
#
# RÈGLE D'OR :
#
#   L'algorithme PROPOSE.
#   L'analyste VALIDE.
#
# COMMENT VALIDER ?
#
#   Examinez chaque liste ci-dessous.
#   Retirez les colonnes mal classées.
#   Ajoutez-les à la bonne liste.
#   Puis relancez convert_column_types().
#
# =====================================================================

def validate_detected_types(
    numeric_cols,
    date_cols,
    category_cols,
    boolean_cols
):
    """
    Affiche les types détectés et guide la validation humaine.

    Cette fonction est un POINT D'ARRÊT dans le pipeline.
    L'analyste doit examiner chaque liste et corriger si nécessaire.

    Parameters
    ----------
    numeric_cols  : list of str
    date_cols     : list of str
    category_cols : list of str
    boolean_cols  : list of str

    Returns
    -------
    tuple : (numeric_cols, date_cols, category_cols, boolean_cols)
        Retourne les listes telles quelles pour permettre
        à l'analyste de les modifier avant la conversion.
    """

    print("\n" + "=" * 60)
    print("HUMAN VALIDATION REQUIRED")
    print("=" * 60)

    print("\n→ Suggested NUMERIC columns :")
    print(f"  {numeric_cols}")

    print("\n→ Suggested DATE columns :")
    print(f"  {date_cols}")

    print("\n→ Suggested CATEGORY columns :")
    print(f"  {category_cols}")

    print("\n→ Suggested BOOLEAN columns :")
    print(f"  {boolean_cols}")

    print("""
─────────────────────────────────────────────────────────
QUESTIONS À SE POSER POUR CHAQUE COLONNE
─────────────────────────────────────────────────────────

① Est-ce vraiment un NOMBRE ?
   age        → OUI  (on calcule une moyenne d'âge)
   salary     → OUI  (on additionne des salaires)
   zipcode    → NON  (code postal, identifiant géographique)
   phone      → NON  (le 0 initial compte)

② Est-ce vraiment une DATE ?
   created_at → OUI
   birth_year → OUI
   id_card_nb → NON  (ressemble à une date, n'en est pas une)

③ Est-ce vraiment une CATÉGORIE ?
   gender     → OUI  (M / F / Other)
   country    → OUI  (50 pays max)
   product_id → NON  (trop de valeurs uniques = identifiant)

④ Est-ce vraiment un BOOLÉEN ?
   is_active  → OUI  (True/False)
   has_paid   → OUI  (Yes/No)
   status     → NON  si status = ["active","pending","closed"]

⑤ Est-ce un IDENTIFIANT ?
   customer_id, employee_id, order_id
   → Conserver en STRING
   → Ne pas convertir en numérique
   → Ne pas utiliser dans des calculs
─────────────────────────────────────────────────────────
""")

    return (
        numeric_cols,
        date_cols,
        category_cols,
        boolean_cols
    )


# =====================================================================
# 9. CONVERSION DES TYPES
# =====================================================================
#
# POURQUOI CETTE ÉTAPE ?
#
#   Après validation humaine, on applique les conversions.
#
#   Un type correct permet :
#   - Des calculs numériques fiables
#   - Des tris de dates corrects
#   - Une mémoire réduite (category vs object)
#   - Un comportement correct dans les modèles ML
#
# =====================================================================

def convert_numeric_columns(df, columns):
    """
    Convertit les colonnes spécifiées en float numérique.

    Parameters
    ----------
    df : pd.DataFrame
    columns : list of str

    Returns
    -------
    pd.DataFrame
    """

    for col in columns:
        # errors="coerce" : valeurs non convertibles → NaN
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def convert_date_columns(df, columns):
    """
    Convertit les colonnes spécifiées en datetime.

    Parameters
    ----------
    df : pd.DataFrame
    columns : list of str

    Returns
    -------
    pd.DataFrame
    """

    for col in columns:
        # errors="coerce" : dates invalides → NaT (Not a Time)
        df[col] = pd.to_datetime(df[col], errors="coerce")

    return df


def convert_category_columns(df, columns):
    """
    Convertit les colonnes spécifiées en type category.

    Étapes de nettoyage appliquées :
        1. Conversion en string
        2. Suppression des espaces en bordure (strip)
        3. Mise en minuscules (lower)
        4. Conversion en category

    Pourquoi minuscules ?
        "France" et "france" sont la même catégorie.
        Sans normalisation, elles sont comptées séparément.

    Parameters
    ----------
    df : pd.DataFrame
    columns : list of str

    Returns
    -------
    pd.DataFrame
    """

    for col in columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()           # supprime les espaces autour
            .str.lower()           # normalise la casse
            .astype("category")    # convertit en type category
        )

    return df


def convert_boolean_columns(df, columns):
    """
    Convertit les colonnes booléennes détectées en type bool Python.

    Mapping appliqué :
        "true" / "yes" / "y" / "1" / 1 → True
        "false" / "no" / "n" / "0" / 0 → False

    Parameters
    ----------
    df : pd.DataFrame
    columns : list of str

    Returns
    -------
    pd.DataFrame
    """

    # Dictionnaire de correspondance vers bool
    BOOL_MAP = {
        "true" : True,  "false": False,
        "yes"  : True,  "no"   : False,
        "y"    : True,  "n"    : False,
        "1"    : True,  "0"    : False,
        1      : True,  0      : False,
    }

    print("\n--- Boolean Conversion ---")

    for col in columns:

        before_dtype = df[col].dtype

        # Normalisation et mapping
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.lower()
            .map(BOOL_MAP)         # applique le dictionnaire
        )

        print(f"  - {col} : {before_dtype} → bool")

    return df


def clean_remaining_text(df):
    """
    Nettoie les colonnes texte restantes non converties.

    Applique uniquement un strip() pour supprimer
    les espaces en début et fin de chaîne.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """

    object_cols = df.select_dtypes(include="object").columns

    for col in object_cols:
        df[col] = df[col].astype(str).str.strip()

    return df


def auto_clean_dataframe(df):
    """
    Orchestre la détection et la conversion automatique des types.

    Étapes :
        1. Détection numérique, date, catégorie, booléen
        2. Affichage pour validation humaine
        3. Conversion

    ⚠ Point critique : modifier les listes AVANT la conversion
      si certaines suggestions sont incorrectes.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """

    print("\n" + "=" * 60)
    print("TYPE DETECTION & CONVERSION")
    print("=" * 60)

    # --- Détection ---
    numeric_cols  = detect_numeric_columns(df)
    date_cols     = detect_date_columns(df)
    category_cols = detect_category_columns(df)
    boolean_cols  = detect_boolean_columns(df)

    # --- Validation humaine ---
    (
        numeric_cols,
        date_cols,
        category_cols,
        boolean_cols
    ) = validate_detected_types(
        numeric_cols,
        date_cols,
        category_cols,
        boolean_cols
    )

    # ─────────────────────────────────────────────────────
    # POINT D'ARRÊT ANALYSTE
    #
    # Ici, l'analyste peut modifier les listes.
    # Exemples :
    #
    #   numeric_cols.remove("zipcode")
    #   boolean_cols.remove("status")
    #   date_cols.append("created_at")
    #
    # ─────────────────────────────────────────────────────

    # --- Conversion ---
    df = convert_boolean_columns(df, boolean_cols)

    # On exclut les booléens déjà traités
    numeric_cols  = [c for c in numeric_cols  if c not in boolean_cols]
    date_cols     = [c for c in date_cols     if c not in boolean_cols]
    category_cols = [c for c in category_cols if c not in boolean_cols]

    df = convert_numeric_columns(df, numeric_cols)
    df = convert_date_columns(df, date_cols)
    df = convert_category_columns(df, category_cols)
    df = clean_remaining_text(df)

    return df


# =====================================================================
# 10. OPTIMISATION MÉMOIRE
# =====================================================================
#
# POURQUOI CETTE ÉTAPE ?
#
#   Pandas utilise par défaut :
#   - int64   pour les entiers   (8 octets par valeur)
#   - float64 pour les décimaux  (8 octets par valeur)
#
#   Si les valeurs sont petites, on peut réduire :
#   - int64  → int8 / int16 / int32  (selon la plage de valeurs)
#   - float64 → float32              (précision réduite mais suffisante)
#
# EXEMPLE CONCRET :
#
#   Dataset de 1 million de lignes, colonne "age" (0-120)
#   int64  → 8 Mo
#   int8   → 1 Mo  (×8 économie)
#
# ATTENTION :
#
#   float32 réduit la précision.
#   Pour des calculs financiers exigeant la précision exacte,
#   garder float64.
#
# QUAND L'UTILISER ?
#
#   - Datasets volumineux (>100k lignes)
#   - Pipelines ML avec beaucoup de features numériques
#   - Quand la RAM est une contrainte
#
# =====================================================================

def optimize_numeric_types(df):
    """
    Réduit l'empreinte mémoire des colonnes numériques.

    Stratégie :
        Entiers (int)   → downcast vers le type entier minimal
        Décimaux (float) → downcast vers float32

    Affiche le gain mémoire avant / après.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame

    Example
    -------
    age : int64 (8 octets) → int8 (1 octet) si valeurs dans [-128, 127]
    """

    print("\n" + "=" * 60)
    print("MEMORY OPTIMIZATION")
    print("=" * 60)

    # Mémoire avant optimisation
    before_mb = df.memory_usage(deep=True).sum() / 1024 ** 2

    for col in df.select_dtypes(include=["integer"]).columns:

        # downcast="integer" : Pandas choisit le plus petit type entier
        # compatible avec les valeurs présentes dans la colonne
        df[col] = pd.to_numeric(df[col], downcast="integer")

    for col in df.select_dtypes(include=["float"]).columns:

        # downcast="float" : passe de float64 à float32
        df[col] = pd.to_numeric(df[col], downcast="float")

    # Mémoire après optimisation
    after_mb = df.memory_usage(deep=True).sum() / 1024 ** 2

    saved = before_mb - after_mb
    ratio = (saved / before_mb * 100) if before_mb > 0 else 0

    print(f"\n  Memory before : {before_mb:.2f} MB")
    print(f"  Memory after  : {after_mb:.2f} MB")
    print(f"  Saved         : {saved:.2f} MB ({ratio:.1f}%)")

    return df


# =====================================================================
# 11. VALIDATION FINALE
# =====================================================================
#
# POURQUOI CETTE ÉTAPE ?
#
#   Après toutes les transformations, on vérifie
#   que le dataset est propre et cohérent.
#
#   C'est le "Quality Check" final avant analyse.
#
# CE QU'ON VÉRIFIE :
#
#   ✓ Plus de valeurs manquantes
#   ✓ Plus de doublons
#   ✓ Types corrects
#   ✓ Mémoire optimisée
#
# =====================================================================

def final_validation(df):
    """
    Effectue un contrôle qualité final sur le dataset nettoyé.

    Affiche :
        - Dimensions finales
        - Valeurs manquantes résiduelles
        - Doublons résiduels
        - Types finaux des colonnes
        - Utilisation mémoire finale

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    None
    """

    print("\n" + "=" * 60)
    print("FINAL VALIDATION")
    print("=" * 60)

    print(f"\nRows    : {df.shape[0]:,}")
    print(f"Columns : {df.shape[1]}")

    # Vérification des manquants résiduels
    print("\n--- Remaining Missing Values ---")
    remaining = df.isnull().sum()
    remaining = remaining[remaining > 0]
    if remaining.empty:
        print("✓ Aucune valeur manquante.")
    else:
        print(remaining)

    # Vérification des doublons résiduels
    print("\n--- Remaining Duplicates ---")
    dup = df.duplicated().sum()
    if dup == 0:
        print("✓ Aucun doublon.")
    else:
        print(f"⚠ {dup} doublon(s) détecté(s).")

    # Types finaux
    print("\n--- Final Data Types ---")
    print(df.dtypes)

    # Mémoire finale
    print("\n--- Final Memory Usage ---")
    memory_mb = df.memory_usage(deep=True).sum() / 1024 ** 2
    print(f"{memory_mb:.2f} MB")

    print("\n" + "=" * 60)
    print("✓ DATASET PRÊT POUR L'ANALYSE")
    print("=" * 60)


# =====================================================================
# 12. PIPELINE MAÎTRE
# =====================================================================
#
# clean_dataset() est le point d'entrée unique.
# Elle orchestre toutes les étapes dans le bon ordre.
#
# =====================================================================

def clean_dataset(df):
    """
    Pipeline complet de nettoyage de données.

    Exécute dans l'ordre :
        1.  inspect_dataframe
        2.  handle_duplicates
        3.  handle_missing_values
        4.  drop_constant_columns
        5.  detect_outliers_iqr         (rapport uniquement)
        6.  auto_clean_dataframe        (détection + conversion des types)
        7.  optimize_numeric_types
        8.  final_validation

    Parameters
    ----------
    df : pd.DataFrame
        Le dataset brut à nettoyer.

    Returns
    -------
    pd.DataFrame
        Le dataset nettoyé, prêt pour l'analyse.

    Example
    -------
    >>> df = pd.read_csv("data.csv")
    >>> df_clean = clean_dataset(df)
    >>> print(df_clean.head())
    """

    # ① Vue d'ensemble
    inspect_dataframe(df)

    # ② Suppression des doublons
    df = handle_duplicates(df)

    # ③ Traitement des valeurs manquantes
    df = handle_missing_values(df)

    # ④ Suppression des colonnes constantes
    df = drop_constant_columns(df)

    # ⑤ Rapport des outliers (LECTURE SEULE — sans modification)
    detect_outliers_iqr(df)

    # ⑥ Détection et conversion des types
    df = auto_clean_dataframe(df)

    # ⑦ Optimisation mémoire
    df = optimize_numeric_types(df)

    # ⑧ Contrôle qualité final
    final_validation(df)

    return df


# =====================================================================
# APRÈS LE NETTOYAGE — QUOI FAIRE ENSUITE ?
# =====================================================================
#
# Le dataset est maintenant propre.
# Il est prêt pour les étapes suivantes :
#
#
# ① EXPLORATION (EDA)
# ─────────────────────────────────────
#
#   df.describe()
#   df.info()
#   df["age"].value_counts()
#
#
# ② VISUALISATION
# ─────────────────────────────────────
#
#   import seaborn as sns
#   import matplotlib.pyplot as plt
#
#   # Distribution d'une variable numérique
#   sns.histplot(df["salary"], kde=True)
#
#   # Détection visuelle des outliers
#   sns.boxplot(x=df["salary"])
#
#   # Corrélations
#   sns.heatmap(df.corr(), annot=True)
#
#   # Scatter : deux variables
#   sns.scatterplot(x="age", y="salary", data=df)
#
#
# ③ STATISTIQUES
# ─────────────────────────────────────
#
#   from scipy.stats import ttest_ind, chi2_contingency
#
#   # Test t entre deux groupes
#   ttest_ind(df[df["gender"]=="m"]["salary"],
#             df[df["gender"]=="f"]["salary"])
#
#   # Test chi2 entre deux catégorielles
#   chi2_contingency(pd.crosstab(df["gender"], df["churn"]))
#
#
# ④ MACHINE LEARNING
# ─────────────────────────────────────
#
#   from sklearn.model_selection import train_test_split
#   from sklearn.preprocessing import StandardScaler
#
#   X = df.drop("target", axis=1)
#   y = df["target"]
#
#   X_train, X_test, y_train, y_test = train_test_split(
#       X, y, test_size=0.2, random_state=42
#   )
#
#   scaler = StandardScaler()
#   X_train_scaled = scaler.fit_transform(X_train)
#
#
# ⑤ DASHBOARDING
# ─────────────────────────────────────
#
#   # Export pour Power BI / Tableau
#   df_clean.to_csv("data_clean.csv", index=False)
#
#   # Export pour Excel
#   df_clean.to_excel("data_clean.xlsx", index=False)
#
#   # Dashboard interactif Streamlit
#   import streamlit as st
#   st.dataframe(df_clean)
#   st.line_chart(df_clean["monthly_revenue"])
#
#
# =====================================================================
# USAGE RAPIDE
# =====================================================================
#
#   import pandas as pd
#   from data_cleaning_pipeline_v2 import (
#       clean_dataset,
#       clean_numeric_strings,
#       detect_outliers_iqr,
#   )
#
#   # Chargement
#   df = pd.read_csv("data.csv")
#
#   # Nettoyage des colonnes monétaires (avant le pipeline)
#   df = clean_numeric_strings(df, columns=["price", "revenue"])
#
#   # Pipeline complet
#   df_clean = clean_dataset(df)
#
#   # Résultat
#   print(df_clean.head())
#   print(df_clean.dtypes)
#
# =====================================================================